# TreeVibes final review — 20 October 2026

Independent GPU experiments; original 131-recipe campaign remains unchanged. RPW is the positive class and healthy the negative class. Validation selection only; final test remains held out. Run setup once, then monitor the detached controller. Existing completed review runs are reused.

## 1. Locate the completed campaign

In [ ]:
from pathlib import Path
import json, os, sys, subprocess
WORK = Path(os.environ.get("TREEVIBES_WINDOWS_WORK", "D:/ML/TreeVibesWindows")).resolve()
OUTPUT = WORK / "final_review_2026-10-20"
ROOT = WORK / "project"
TF_PYTHON = WORK / "runtimes/tf216-cpu/Scripts/python.exe"
GPU_PYTHON = WORK / "runtimes/torch271-cu128/Scripts/python.exe"
DATASET_ROOT = None  # Reuse/download the Kaggle WAV dataset automatically.
ADDITIONAL_FEATURES = True
assert TF_PYTHON.is_file() and GPU_PYTHON.is_file(), "Use the existing Windows training setup."
original = json.loads((WORK / "experiments/campaign_summary.json").read_text())
assert original["state"] in {"completed", "completed_with_failures"}, "Do not interrupt the original campaign."
OUTPUT.mkdir(parents=True, exist_ok=True)
for name in ["tools/scripts", "configs", "checkpoints", "plots", "logs", "audit", "cache"]:
    (OUTPUT / name).mkdir(parents=True, exist_ok=True)
settings = dict(original=str(WORK/"experiments"), output=str(OUTPUT), project=str(ROOT),
    tools_parent=str(OUTPUT/"tools"), tf_python=str(TF_PYTHON), torch_python=str(GPU_PYTHON),
    dataset_root=DATASET_ROOT, additional_features=ADDITIONAL_FEATURES)
progress_path = OUTPUT / "progress.json"
progress = json.loads(progress_path.read_text()) if progress_path.exists() else {}
ACTIVE = False
controller = OUTPUT / "controller.json"
if controller.exists() and progress.get("state") == "running":
    pid = json.loads(controller.read_text())["pid"]
    probe = subprocess.run(["tasklist", "/FI", f"PID eq {pid}", "/FO", "CSV", "/NH"], capture_output=True, text=True)
    ACTIVE = f'"{pid}"' in probe.stdout
print("Existing campaign:", original["completed_runs"], "/", original["planned_runs"])
print("Review:", progress, "controller active:", ACTIVE)


## 2. Prepare isolated review tools

Uses the existing Torch CUDA and TensorFlow CPU workers; new packages go into the review folder.

In [ ]:
sources = {'review_campaign.py': '"""Run the isolated professor-review study without modifying the original campaign.\n\nOnly train/validation arrays are loaded. The existing allocation and accepted recording\nIDs define every comparison; final-test labels remain outside this campaign.\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport traceback\nimport numpy as np\n\n\ndef read_json(path):\n    """Read UTF-8 artifacts consistently on native Windows and the local review host."""\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef save_json(path, value):\n    """Commit evidence atomically so interrupted workers cannot appear completed."""\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    temp=path.with_suffix(path.suffix+\'.tmp\')\n    temp.write_text(json.dumps(value,indent=2,allow_nan=False),encoding=\'utf-8\');temp.replace(path)\n\n\ndef cohort_digest(ids):\n    """Fingerprint ordered recording IDs rather than comparing only sample counts."""\n    return hashlib.sha256(json.dumps(list(map(str,ids)),ensure_ascii=True).encode()).hexdigest()\n\n\ndef fit_scaling(x):\n    """Fit channel scaling on training frames; keep constant channels numerically safe."""\n    mean=x.mean(axis=(0,1),dtype=np.float64).astype(\'float32\')\n    scale=x.std(axis=(0,1),dtype=np.float64).astype(\'float32\')\n    return mean,np.where(scale<1e-6,1.,scale).astype(\'float32\')\n\n\ndef review_metrics(labels, probabilities, threshold):\n    """Retain RPW false negatives and healthy false alarms alongside headline metrics."""\n    from treevibes.evaluate import evaluate_probabilities\n    m=evaluate_probabilities(np.asarray(labels),np.asarray(probabilities),threshold)\n    tn,fp=m[\'confusion\'][0];fn,tp=m[\'confusion\'][1]\n    return m|dict(false_negatives=fn,false_positives=fp,fpr=fp/max(tn+fp,1))\n\n\ndef resolve_audio(root, identifier):\n    """Resolve an exact relative recording identity, never an ambiguous filename match."""\n    root=Path(root)\n    for prefix in [root,root/\'field/field/train\']:\n        path=prefix/identifier\n        if path.is_file():return path\n    raise FileNotFoundError(f\'Original WAV is missing: {identifier}\')\n\n\n\ndef read_cache(snapshot, identifier):\n    """Verify existing cache bytes while opening only training/validation arrays."""\n    for path in (Path(snapshot)/\'cache\').glob(\'*/contract.json\'):\n        contract=read_json(path)\n        if contract[\'dataset_id\']!=identifier:continue\n        payload=path.parent/\'features.npz\'\n        digest=hashlib.sha256()\n        with payload.open(\'rb\') as stream:\n            for chunk in iter(lambda:stream.read(1024*1024),b\'\'):digest.update(chunk)\n        if digest.hexdigest()!=contract[\'payload_sha256\']:raise ValueError(\'Original feature cache changed\')\n        with np.load(payload,allow_pickle=False) as archive:\n            data={key:archive[key] for key in [\'X_train\',\'y_train\',\'ids_train\',\'X_val\',\'y_val\',\'ids_val\']}\n        return data|dict(contract=contract)\n    raise ValueError(f\'No matching cache: {identifier}\')\n\ndef audit(settings):\n    """Freeze the canonical 131-recipe results, verifying checkpoint and prediction integrity."""\n    import pandas as pd\n    from scripts.windows_support import valid_imported_run\n    old=Path(settings[\'original\']);out=Path(settings[\'output\']);out.mkdir(parents=True,exist_ok=True)\n    summary=read_json(old/\'campaign_summary.json\')\n    if summary[\'state\'] not in {\'completed\',\'completed_with_failures\'}:\n        raise RuntimeError(\'Original campaign is active or incomplete; audit only, do not start workers.\')\n    save_json(out/\'audit/original_campaign_summary.json\',summary)\n    primary=summary[\'primary_dataset_id\'];data=read_cache(old,primary)\n    ids={part:cohort_digest(data[f\'ids_{part}\']) for part in [\'train\',\'val\']}\n    rows=[]\n    for entry in summary[\'runs\']:\n        directory=Path(entry[\'run_dir\']);valid=valid_imported_run(directory)\n        c=read_json(directory/\'config.json\');r=read_json(directory/\'training_result.json\')\n        contract=read_json(directory/\'input_contract.json\')\n        history=r.get(\'history\',[])\n        with np.load(directory/\'validation_predictions.npz\',allow_pickle=False) as p:\n            labels=p[\'labels\'];probs=p[\'probabilities\']\n        metrics=review_metrics(labels,probs,r[\'validation\'][\'threshold\'])\n        same=contract[\'dataset_id\']==primary\n        # Different feature caches may still share exactly the same recordings/order.\n        if not same:\n            other=read_cache(old,contract[\'dataset_id\'])\n            same=all(cohort_digest(other[f\'ids_{part}\'])==ids[part] for part in [\'train\',\'val\'])\n        row=dict(id=c[\'id\'],status=\'completed\' if valid else \'invalid_artifacts\',run_dir=str(directory),\n                 family=c[\'family\'],target=c[\'target\'],dataset_id=contract[\'dataset_id\'],\n                 feature_shape=f"{c[\'features\'][\'frames\']}x{c[\'features\'][\'coefficients\']}",\n                 seed=c[\'seed\'],epochs_completed=r.get(\'epochs_completed\',0),\n                 best_epoch=min(history,key=lambda x:x[\'val_loss\'])[\'epoch\'] if history else None,\n                 same_evaluation_cohort=same,validation_ids_hash=ids[\'val\'] if same else None,**metrics)\n        rows.append(row)\n        target=out/\'audit/models\'/c[\'id\'];target.mkdir(parents=True,exist_ok=True)\n        for name in [\'config.json\',\'input_contract.json\',\'training_result.json\',\'validation_predictions.npz\',\'provenance.json\',\'epoch_state.json\']:\n            if (directory/name).exists():shutil.copy2(directory/name,target/name)\n    if len({r[\'id\'] for r in rows})!=summary[\'planned_runs\']:\n        raise ValueError(\'Canonical campaign recipe count does not match recorded rows\')\n    pd.DataFrame(rows).to_csv(out/\'previous_results.csv\',index=False)\n    save_json(out/\'audit/previous_results.json\',rows)\n    candidates=[r for r in rows if r[\'status\']==\'completed\' and r[\'same_evaluation_cohort\'] and (Path(r[\'run_dir\'])/\'best.keras\').exists()]\n    rank=lambda r:(r[\'f1\'],r[\'recall\'],-r[\'false_positives\'])\n    server=max([r for r in candidates if r[\'target\']==\'remote\'],key=rank)\n    local=max([r for r in candidates if r[\'target\']==\'local\' and r[\'feature_shape\']==\'30x13\'],key=rank)\n    selection=dict(server=server,local=local,primary_dataset_id=primary,cohort=ids,\n                   validation_count=len(data[\'y_val\']),train_count=len(data[\'y_train\']),\n                   classes=[\'healthy\',\'RPW\'],test_set_evaluated=False,\n                   grouping=data[\'contract\'][\'split\'][\'grouping\'])\n    save_json(out/\'audit/selection.json\',selection)\n    # Preserve the allocation, including held-out identities, without opening test arrays.\n    save_json(out/\'audit/original_split.json\',data[\'contract\'][\'split\'])\n    save_json(out/\'audit/original_input_contract.json\',data[\'contract\'])\n    save_json(out/\'audit/original_exports.json\',summary[\'exports\'])\n    print(\'AUDIT\',json.dumps({k:v[\'id\'] if isinstance(v,dict) and \'id\' in v else v for k,v in selection.items()}),flush=True)\n    return selection\n\n\ndef cache_baseline(settings):\n    """Reuse cached MFCC bytes and exact training statistics for model continuation."""\n    out=Path(settings[\'output\']);selection=read_json(out/\'audit/selection.json\')\n    data=read_cache(Path(settings[\'original\']),selection[\'primary_dataset_id\'])\n    destination=out/\'cache/mfcc13\';destination.mkdir(parents=True,exist_ok=True)\n    c=data[\'contract\'];stats=c[\'normalization\']\n    for part in [\'train\',\'val\']:\n        np.save(destination/f\'X_{part}.npy\',data[f\'X_{part}\'])\n        np.save(destination/f\'y_{part}.npy\',data[f\'y_{part}\'])\n        np.save(destination/f\'ids_{part}.npy\',data[f\'ids_{part}\'])\n    save_json(destination/\'contract.json\',dict(representation=\'mfcc13\',shape=[30,13],\n        features=c[\'features\'],mean=stats[\'mean\'],scale=stats[\'scale\'],\n        train_ids_hash=selection[\'cohort\'][\'train\'],validation_ids_hash=selection[\'cohort\'][\'val\'],\n        original_dataset_id=c[\'dataset_id\'],source=\'original_feature_cache\'))\n    return destination\n\n\ndef prepare_features(settings):\n    """Extract comparable features and high-information images from the same accepted WAVs."""\n    import librosa\n    from scipy.signal import butter,filtfilt\n    from scipy.ndimage import zoom\n    out=Path(settings[\'output\']);selection=read_json(out/\'audit/selection.json\')\n    base=out/\'cache/mfcc13\';c=read_json(base/\'contract.json\')[\'features\']\n    identifiers={p:np.load(base/f\'ids_{p}.npy\',allow_pickle=False) for p in [\'train\',\'val\']}\n    names=dict(mfcc20=(30,20),mfcc40=(30,40),mel32=(30,32),mel64=(30,64),pcen64=(30,64),stft=(30,1334),image_mel=(299,299))\n    complete=out/\'cache/features_ready.json\'\n    if complete.exists():return read_json(complete)\n    if settings.get(\'dataset_root\'):\n        root=Path(settings[\'dataset_root\'])\n    else:\n        import kagglehub\n        root=Path(kagglehub.dataset_download(\'potamitis/treevibes\'))\n    save_json(out/\'audit/audio_source.json\',dict(root=str(root),sample_rate=8000,screen_seconds=10,\n                                               unrestricted_seconds_cap=30))\n    manifest=read_json(next(p for p in (Path(settings[\'original\'])/\'cache\').glob(\'*/manifest.json\')\n                            if read_json(p.parent/\'contract.json\')[\'dataset_id\']==selection[\'primary_dataset_id\']))\n    expected={r[\'id\']:r[\'sha256\'] for r in manifest}\n    arrays={}\n    for name,shape in names.items():\n        dest=out/\'cache\'/name;dest.mkdir(exist_ok=True)\n        arrays[name]={}\n        for part in [\'train\',\'val\']:\n            arrays[name][part]=np.lib.format.open_memmap(dest/f\'X_{part}.npy\',mode=\'w+\',dtype=\'float16\' if name==\'image_mel\' else \'float32\',shape=(len(identifiers[part]),*shape))\n            for prefix in [\'y\',\'ids\']:shutil.copy2(base/f\'{prefix}_{part}.npy\',dest/f\'{prefix}_{part}.npy\')\n    b,a=butter(4,[100,2500],btype=\'bandpass\',fs=8000)\n    first=True;done=0;start=time.monotonic()\n    for part in [\'train\',\'val\']:\n        for index,identifier in enumerate(identifiers[part]):\n            path=resolve_audio(root,str(identifier))\n            if hashlib.sha256(path.read_bytes()).hexdigest()!=expected[str(identifier)]:\n                raise ValueError(f\'WAV identity changed: {identifier}\')\n            signal,_=librosa.load(path,sr=8000,mono=True)\n            clip=np.pad(signal[:80000],(0,max(0,80000-len(signal))))\n            filtered=filtfilt(b,a,clip)\n            fft=80000//30\n            power=abs(librosa.stft(filtered,n_fft=fft,hop_length=fft,win_length=fft,center=False))**2\n            mel={n:librosa.feature.melspectrogram(S=power,sr=8000,n_fft=fft,n_mels=n,fmin=0,fmax=4000,norm=\'slaney\') for n in [32,64,128]}\n            logs={n:librosa.power_to_db(mel[n],ref=1.,amin=1e-10,top_db=80) for n in mel}\n            for n in [20,40]:arrays[f\'mfcc{n}\'][part][index]=librosa.feature.mfcc(S=logs[128],n_mfcc=n).T\n            for n in [32,64]:arrays[f\'mel{n}\'][part][index]=logs[n].T\n            pcen=librosa.pcen(np.sqrt(mel[64])*32768,sr=8000,hop_length=fft,gain=.98,bias=2,power=.5,time_constant=.4,eps=1e-6)\n            arrays[\'pcen64\'][part][index]=pcen.T\n            arrays[\'stft\'][part][index]=librosa.power_to_db(power,ref=1.,top_db=80).T\n            # Full original audio (up to 30 s), retaining short-time events before resizing.\n            full=filtfilt(b,a,signal[:240000])\n            high=librosa.feature.melspectrogram(y=full,sr=8000,n_fft=512,hop_length=128,n_mels=128,fmin=0,fmax=4000)\n            image=librosa.power_to_db(high,ref=np.max,top_db=80)\n            image=zoom(image,(299/image.shape[0],299/image.shape[1]),order=1)\n            arrays[\'image_mel\'][part][index]=image.astype(\'float16\')\n            if first:\n                np.savez_compressed(out/\'plots/same_recording_features.npz\',mfcc=np.load(base/\'X_train.npy\',mmap_mode=\'r\')[0],mel=logs[64].T,pcen=pcen.T,recording=str(identifier))\n                first=False\n            done+=1\n            if done%100==0:print(f\'FEATURES {done}/2070, {(time.monotonic()-start)/60:.1f} min\',flush=True)\n    for name,parts in arrays.items():\n        for array in parts.values():array.flush()\n        if name==\'image_mel\':mean,scale=0.,1.\n        else:\n            mean,scale=fit_scaling(parts[\'train\']);mean=mean.tolist();scale=scale.tolist()\n        save_json(out/\'cache\'/name/\'contract.json\',dict(representation=name,shape=list(names[name]),mean=mean,scale=scale,\n            train_ids_hash=selection[\'cohort\'][\'train\'],validation_ids_hash=selection[\'cohort\'][\'val\'],\n            features=dict(sample_rate=8000,seconds=30 if name==\'image_mel\' else 10,bandpass=[100,2500],filter=\'filtfilt_order4\',\n                fft_size=512 if name==\'image_mel\' else fft,hop=128 if name==\'image_mel\' else fft,center=False if name!=\'image_mel\' else True,\n                power=2,mel_norm=\'slaney\',pcen_amplitude_multiplier=32768 if name==\'pcen64\' else None,\n                pcen=dict(gain=.98,bias=2,power=.5,time_constant=.4,eps=1e-6) if name==\'pcen64\' else None,\n                image_resize=\'bilinear frequency/time to 299x299; replicate one channel to RGB; dB [-80,0] to [-1,1]\' if name==\'image_mel\' else None)))\n    save_json(complete,dict(representations=list(names),recordings=done,elapsed_seconds=time.monotonic()-start))\n    return read_json(complete)\n\n\ndef bridge_job(request):\n    """Serialize Keras graphs and INT8 models in the CPU worker, away from CUDA allocations."""\n    import tensorflow as tf\n    from scripts.torch_bridge import dump_model,apply_weights\n    from treevibes.models import build_model\n    directory=Path(request[\'directory\']);directory.mkdir(parents=True,exist_ok=True)\n    if request[\'action\']==\'bridge_dump\':\n        c=request[\'config\'];tf.keras.utils.set_random_seed(c[\'seed\'])\n        model=tf.keras.models.load_model(request[\'model\'],compile=False) if request.get(\'model\') else build_model(c)\n        dump_model(model,directory);model.save(directory/\'template.keras\')\n    elif request[\'action\']==\'bridge_restore\':\n        model=tf.keras.models.load_model(directory/\'template.keras\',compile=False);apply_weights(model,request[\'weights\'])\n        with np.load(directory.parent/\'bridge_parity.npz\') as p:\n            np.testing.assert_allclose(model(p[\'features\'][...,None],training=False).numpy().ravel(),p[\'logits\'],rtol=2e-4,atol=5e-5)\n        model.save(request[\'model\'])\n    elif request[\'action\']==\'int8\':\n        from treevibes.export import tflite_predict,_operators,_details\n        from treevibes.evaluate import select_threshold\n        model=tf.keras.models.load_model(request[\'model\'],compile=False)\n        cache=Path(request[\'cache\']);contract=read_json(cache/\'contract.json\')\n        x=(np.load(cache/\'X_train.npy\')-np.array(contract[\'mean\']))/np.array(contract[\'scale\'])\n        xv=(np.load(cache/\'X_val.npy\')-np.array(contract[\'mean\']))/np.array(contract[\'scale\'])\n        x=x.astype(\'float32\');xv=xv.astype(\'float32\');yv=np.load(cache/\'y_val.npy\')\n        converter=tf.lite.TFLiteConverter.from_keras_model(model);converter.optimizations=[tf.lite.Optimize.DEFAULT]\n        indices=np.random.default_rng(42).permutation(len(x))[:128]\n        def representative():\n            """Calibrate only on training observations."""\n            for i in indices:yield [x[i:i+1,...,None]]\n        converter.representative_dataset=representative\n        converter.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]\n        converter.inference_input_type=tf.int8;converter.inference_output_type=tf.int8\n        contents=converter.convert();target=directory/\'model.tflite\';target.write_bytes(contents)\n        interpreter=tf.lite.Interpreter(model_content=contents);interpreter.allocate_tensors()\n        if any(np.issubdtype(t[\'dtype\'],np.floating) for t in interpreter.get_tensor_details()):raise ValueError(\'Float tensors remain in INT8 export\')\n        p=tflite_predict(target,xv);threshold=select_threshold(yv,p)\n        np.savez_compressed(directory/\'validation_predictions.npz\',labels=yv,probabilities=p,ids=np.load(cache/\'ids_val.npy\'))\n        meta=dict(model_bytes=len(contents),model_sha256=hashlib.sha256(contents).hexdigest(),board_verified=False,\n                  operators=_operators(contents),input=_details(interpreter.get_input_details()[0]),\n                  output=_details(interpreter.get_output_details()[0]),**review_metrics(yv,p,threshold))\n        save_json(directory/\'export.json\',meta)\n    else:raise ValueError(\'Unknown CPU bridge action\')\n\n\ndef worker(settings, request, framework=\'torch\'):\n    """Run one bounded worker, retaining logs and releasing framework memory afterward."""\n    out=Path(settings[\'output\']);name=request.get(\'id\',request[\'action\'])\n    request_path=out/\'configs\'/f\'{name}_{request["action"]}.json\';save_json(request_path,request)\n    env=os.environ.copy();env.update(PYTHONPATH=os.pathsep.join([settings[\'tools_parent\'],settings[\'project\'],str(out/\'dependencies\')]),\n        PYTHONUTF8=\'1\',PYTHONIOENCODING=\'utf-8\',MPLBACKEND=\'Agg\',TF_USE_LEGACY_KERAS=\'1\',\n        TF_NUM_INTEROP_THREADS=\'1\',TF_NUM_INTRAOP_THREADS=\'1\',OMP_NUM_THREADS=\'2\',OPENBLAS_NUM_THREADS=\'2\',\n        HF_HOME=str(out/\'pretrained_cache\'))\n    if framework==\'tf\':env[\'CUDA_VISIBLE_DEVICES\']=\'-1\'\n    script=Path(__file__).with_name(\'review_torch.py\') if framework==\'torch\' else Path(__file__)\n    command=[settings[f\'{framework}_python\'],str(script),\'--request\',str(request_path)]\n    from scripts.windows_support import run_logged\n    run_logged(command,out/\'logs\'/f\'{name}_{request["action"]}.log\',env=env,stream=True,cwd=settings[\'project\'])\n\n\ndef run_campaign(settings):\n    """Prioritize finished evidence: extensions, core features, unrestricted model, then follow-ups."""\n    out=Path(settings[\'output\']);out.mkdir(parents=True,exist_ok=True)\n    selection=audit(settings);cache_baseline(settings)\n    failures=[]\n    def train(spec):\n        """Skip intact completed review runs and record failures without discarding other evidence."""\n        directory=out/\'checkpoints\'/spec[\'id\'];spec=spec|dict(directory=str(directory),output=str(out))\n        if (directory/\'result.json\').exists():print(\'REUSE\',spec[\'id\'],flush=True);return\n        save_json(out/\'progress.json\',dict(state=\'running\',active=spec[\'id\'],failures=failures))\n        try:\n            if spec.get(\'family\')!=\'inception_resnet_v2\':\n                graph=directory/\'graph\';graph.mkdir(parents=True,exist_ok=True)\n                worker(settings,dict(action=\'bridge_dump\',id=spec[\'id\'],directory=str(graph),config=spec[\'model_config\'],\n                                     model=str(Path(spec[\'origin\'])/\'best.keras\') if spec.get(\'origin\') else None),framework=\'tf\')\n            worker(settings,spec|dict(action=\'train\'))\n        except Exception as error:\n            failures.append(dict(id=spec[\'id\'],error=str(error)));save_json(directory/\'failure.json\',failures[-1])\n            print(\'RECORDED FAILURE\',spec[\'id\'],str(error)[-1000:],flush=True)\n    for route in [\'server\',\'local\']:\n        previous=selection[route];c=read_json(Path(previous[\'run_dir\'])/\'config.json\')\n        train(dict(id=f\'extend_{route}\',kind=\'extension\',family=c[\'family\'],model_config=c,\n                   origin=previous[\'run_dir\'],cache=str(out/\'cache/mfcc13\'),epochs=200,patience=20,route=route,\n                   learning_rate_factor=1.,seed=c[\'seed\'],batch_size=c[\'batch_size\']))\n    prepare_features(settings)\n    baseline=next(r for r in read_json(out/\'audit/previous_results.json\') if r[\'id\']==\'cnn_w16\')\n    c=read_json(Path(baseline[\'run_dir\'])/\'config.json\')\n    def screen(name,epochs=60,width=16,kind=\'feature_screen\'):\n        """Use identical CNN channels, seed, optimizer and audio coverage across screens."""\n        contract=read_json(out/\'cache\'/name/\'contract.json\');model=c|dict(width=width)\n        model[\'features\']=model[\'features\']|dict(frames=contract[\'shape\'][0],coefficients=contract[\'shape\'][1])\n        train(dict(id=f\'{kind}_{name}\',kind=kind,family=\'cnn\',model_config=model,\n                   cache=str(out/\'cache\'/name),epochs=epochs,patience=15 if epochs==60 else 20,\n                   route=\'server\',seed=42,batch_size=16))\n    for name in [\'mfcc13\',\'mel64\',\'pcen64\']:screen(name)\n    train(dict(id=\'unrestricted_inception_resnet_v2\',kind=\'unrestricted\',family=\'inception_resnet_v2\',\n               cache=str(out/\'cache/image_mel\'),epochs=40,head_epochs=5,patience=10,seed=42,\n               batch_size=2,accumulation=8,route=\'unrestricted\',learning_rate=1e-4,\n               backbone=\'inception_resnet_v2.tf_in1k\',pretrained=True))\n    if settings.get(\'additional_features\',True):\n        for name in [\'mfcc20\',\'mfcc40\',\'mel32\',\'stft\']:screen(name)\n    screens=[read_json(p) for p in (out/\'checkpoints\').glob(\'feature_screen_*/result.json\') if \'mfcc13\' not in str(p)]\n    # Advance two non-baseline features using the declared recall/F1/false-alarm ranking.\n    for r in sorted(screens,key=lambda r:(r[\'recall\'],r[\'f1\'],-r[\'fpr\']),reverse=True)[:2]:\n        screen(r[\'representation\'],epochs=120,width=32,kind=\'feature_full\')\n    save_json(out/\'progress.json\',dict(state=\'completed\' if not failures else \'completed_with_failures\',failures=failures))\n    print(\'REVIEW TRAINING COMPLETE\',json.dumps(failures),flush=True)\n\n\ndef main():\n    """Separate CPU conversion jobs from the campaign controller and CUDA fitting workers."""\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--settings\',type=Path);parser.add_argument(\'--request\',type=Path)\n    args=parser.parse_args()\n    if args.request:bridge_job(read_json(args.request))\n    elif args.settings:run_campaign(read_json(args.settings))\n    else:parser.error(\'Provide --settings or --request\')\n\n\nif __name__==\'__main__\':main()\n', 'review_torch.py': '"""CUDA-only review training with durable epochs, fixed cohorts and measured resource use."""\nimport argparse\nimport copy\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport time\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom scripts.review_campaign import read_json,save_json,review_metrics,cohort_digest\n\n\ndef load_arrays(cache, image=False):\n    """Memory-map image inputs and never open final-test labels or features."""\n    cache=Path(cache);contract=read_json(cache/\'contract.json\')\n    data={name:np.load(cache/f\'{name}.npy\',mmap_mode=\'r\',allow_pickle=False)\n          for name in [\'X_train\',\'y_train\',\'ids_train\',\'X_val\',\'y_val\',\'ids_val\']}\n    for part in [\'train\',\'val\']:\n        key=\'train_ids_hash\' if part==\'train\' else \'validation_ids_hash\'\n        if cohort_digest(data[f\'ids_{part}\'])!=contract[key]:raise ValueError(\'Recording cohort changed\')\n        if not image:\n            data[f\'X_{part}\']=((data[f\'X_{part}\']-np.array(contract[\'mean\'],dtype=\'float32\'))/\n                               np.array(contract[\'scale\'],dtype=\'float32\')).astype(\'float32\')\n    return data,contract\n\n\ndef input_batch(x, indices, image):\n    """Replicate the spectrogram channel and apply the pretrained model\'s [-1,1] input scale."""\n    value=torch.as_tensor(np.array(x[indices],copy=True),dtype=torch.float32,device=\'cuda\')\n    return ((value[:,None]+80)/40-1).repeat(1,3,1,1) if image else value\n\n\ndef predictions(model,x,batch,image):\n    """Bound inference allocations independently of training accumulation or dataset size."""\n    values=[];model.eval()\n    with torch.no_grad():\n        for start in range(0,len(x),batch):\n            xb=input_batch(x,slice(start,start+batch),image)\n            with torch.autocast(\'cuda\',enabled=image):z=model(xb).float()\n            values.append(z.cpu().numpy().ravel())\n    return np.concatenate(values)\n\n\ndef cpu_state(model):\n    """Detach checkpoints from CUDA so best-state bookkeeping cannot double GPU allocation."""\n    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\n\n\ndef train(spec):\n    """Continue original optimizer state or train controlled features/pretrained InceptionResNet-V2."""\n    from scipy.special import expit\n    from treevibes.evaluate import select_threshold\n    from scripts.torch_adapter import GraphModel,save_keras_weights\n    from scripts.review_campaign import worker\n    if not torch.cuda.is_available():raise RuntimeError(\'CUDA is required; no silent CPU fallback.\')\n    torch.set_num_threads(2);random.seed(spec[\'seed\']);np.random.seed(spec[\'seed\']);torch.manual_seed(spec[\'seed\'])\n    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.allow_tf32=False;torch.backends.cuda.matmul.allow_tf32=False\n    directory=Path(spec[\'directory\']);directory.mkdir(parents=True,exist_ok=True)\n    save_json(directory/\'config.json\',spec)\n    image=spec[\'family\']==\'inception_resnet_v2\';data,contract=load_arrays(spec[\'cache\'],image)\n    if image:\n        import timm\n        model=timm.create_model(spec[\'backbone\'],pretrained=spec[\'pretrained\'],num_classes=1)\n        # Verify the selected pretrained input normalization, rather than assuming ImageNet defaults.\n        cfg=model.pretrained_cfg\n        if tuple(cfg.get(\'mean\',()))!=(.5,.5,.5) or tuple(cfg.get(\'std\',()))!=(.5,.5,.5):\n            raise ValueError(\'Unexpected pretrained input scale; update the explicit image transform.\')\n        save_json(directory/\'pretrained_config.json\',{k:v for k,v in cfg.items() if isinstance(v,(str,int,float,list,tuple,bool,type(None)))})\n    else:model=GraphModel(directory/\'graph\')\n    model=model.cuda();optimizer=torch.optim.Adam(model.parameters(),lr=spec.get(\'learning_rate\',spec.get(\'model_config\',{}).get(\'learning_rate\',1e-3)),eps=1e-7)\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=image)\n    history=[];start_epoch=0;best_loss=1e30;best_weights=cpu_state(model);stale=0\n    original_history=[];origin_note=None\n    last=directory/\'last.pt\'\n    if last.exists():\n        saved=torch.load(last,map_location=\'cpu\',weights_only=False)\n        if saved[\'spec\']!=spec:raise ValueError(\'Resume specification changed; use a separate run ID.\')\n        model.load_state_dict(saved[\'model\']);optimizer.load_state_dict(saved[\'optimizer\']);scaler.load_state_dict(saved[\'scaler\'])\n        history=saved[\'history\'];start_epoch=saved[\'epoch\'];best_loss=saved[\'best_loss\'];stale=saved[\'stale\']\n        best_weights=torch.load(directory/\'best.pt\',map_location=\'cpu\',weights_only=True)\n    elif spec.get(\'origin\'):\n        origin=Path(spec[\'origin\']);state=read_json(origin/\'epoch_state.json\')\n        if state.get(\'backend\')!=\'torch_cuda_v1\':raise ValueError(\'Continuation needs a CUDA optimizer checkpoint.\')\n        saved=torch.load(origin/state[\'checkpoint\'],map_location=\'cpu\',weights_only=False)\n        best=torch.load(origin/state[\'best_checkpoint\'],map_location=\'cpu\',weights_only=False)\n        model.load_state_dict(saved[\'model\']);optimizer.load_state_dict(saved[\'optimizer\'])\n        # Optimizer moments/rate persist; only the explicit LR ablation changes the rate.\n        for group in optimizer.param_groups:group[\'lr\']*=spec.get(\'learning_rate_factor\',1.)\n        best_weights=best[\'model\'];best_loss=state[\'best_loss\'];start_epoch=state[\'epoch\'];history=state[\'history\'].copy()\n        original_history=history.copy();origin_note=dict(epoch=start_epoch,optimizer_restored=True,\n            previous_stale=state[\'stale\'],new_patience=spec[\'patience\'],stale_reset_for_continuation=True)\n    # Adam state tensors must follow parameters when checkpoints were read on CPU.\n    for state in optimizer.state.values():\n        for key,value in state.items():\n            if torch.is_tensor(value):state[key]=value.cuda()\n    torch.save(best_weights,directory/\'best.pt\')\n    classes=np.bincount(data[\'y_train\'],minlength=2)\n    weighted=bool(spec.get(\'model_config\',{}).get(\'class_weight\',False))\n    class_weights=torch.tensor(len(data[\'y_train\'])/(2*classes) if weighted else [1,1],device=\'cuda\',dtype=torch.float32)\n    started=time.monotonic();torch.cuda.reset_peak_memory_stats();head_epochs=spec.get(\'head_epochs\',0)\n    switched=False\n    for epoch in range(start_epoch,spec[\'epochs\']):\n        if stale>=spec[\'patience\']:break\n        torch.manual_seed(spec[\'seed\']+epoch);rng=np.random.default_rng(spec[\'seed\']+epoch)\n        if image:\n            fine=epoch>=head_epochs\n            for name,p in model.named_parameters():\n                p.requires_grad=name.startswith(\'classif\') or (fine and name.startswith((\'repeat_2.\',\'block8.\',\'conv2d_7b.\')))\n            if epoch==head_epochs and not switched:\n                # Fine-tuning has its own low learning rate, never an epoch-limit ablation.\n                for group in optimizer.param_groups:group[\'lr\']=1e-5\n                stale=0;switched=True\n        model.train()\n        if image:\n            # Frozen BatchNorm statistics must not drift during head training or upper-block tuning.\n            for module in model.modules():\n                if isinstance(module,nn.modules.batchnorm._BatchNorm):module.eval()\n        order=rng.permutation(len(data[\'y_train\']));loss_total=0.;epoch_start=time.monotonic();rate=optimizer.param_groups[0][\'lr\']\n        batch=spec[\'batch_size\'];accum=spec.get(\'accumulation\',1);optimizer.zero_grad(set_to_none=True)\n        steps=list(range(0,len(order),batch))\n        for step,start in enumerate(steps):\n            indices=order[start:start+batch];xb=input_batch(data[\'X_train\'],indices,image)\n            yb=torch.as_tensor(np.array(data[\'y_train\'][indices],copy=True),device=\'cuda\',dtype=torch.float32).reshape(-1,1)\n            c=spec.get(\'model_config\',{})\n            if c.get(\'noise\'):xb+=torch.as_tensor(rng.normal(0,c[\'noise\'],xb.shape),device=\'cuda\',dtype=torch.float32)\n            if c.get(\'time_mask\'):\n                for row in xb:\n                    begin=rng.integers(0,row.shape[0]-c[\'time_mask\']+1);row[begin:begin+c[\'time_mask\']]=0\n            # Weight each microbatch by observations in its accumulation window, including the last short window.\n            window_start=(step//accum)*accum;window_end=min(window_start+accum,len(steps))\n            window_count=sum(min(batch,len(order)-steps[j]) for j in range(window_start,window_end))\n            with torch.autocast(\'cuda\',enabled=image):\n                logits=model(xb);loss=nn.functional.binary_cross_entropy_with_logits(logits,yb,reduction=\'none\')\n                loss=(loss*class_weights[yb.long()]).mean()\n            if not torch.isfinite(loss):raise ValueError(\'Nonfinite training loss\')\n            scaler.scale(loss*len(indices)/window_count).backward();loss_total+=float(loss.detach())*len(indices)\n            if step+1==window_end:\n                scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True)\n        z=predictions(model,data[\'X_val\'],batch if image else 64,image)\n        val_loss=float(np.mean(np.logaddexp(0,z)-data[\'y_val\']*z))\n        if not np.isfinite(val_loss):raise ValueError(\'Nonfinite validation loss\')\n        p=expit(z);m=review_metrics(data[\'y_val\'],p,.5)\n        improved=val_loss<best_loss;stale=0 if improved else stale+1\n        if improved:\n            best_loss=val_loss;best_weights=cpu_state(model)\n            temp=directory/\'best.tmp.pt\';torch.save(best_weights,temp);temp.replace(directory/\'best.pt\')\n        row=dict(epoch=epoch+1,train_loss=loss_total/len(order),val_loss=val_loss,learning_rate=rate,\n                 recall_at_05=m[\'recall\'],f1_at_05=m[\'f1\'],seconds=time.monotonic()-epoch_start,\n                 stage=\'fine_tune\' if image and epoch>=head_epochs else \'head\' if image else \'supervised\')\n        history.append(row)\n        if stale and stale%5==0:\n            for group in optimizer.param_groups:group[\'lr\']=max(group[\'lr\']*.5,1e-6)\n        payload=dict(spec=spec,model=cpu_state(model),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),\n                     epoch=epoch+1,history=history,best_loss=best_loss,stale=stale)\n        temp=directory/\'last.tmp.pt\';torch.save(payload,temp);temp.replace(last)\n        save_json(directory/\'history.json\',history)\n        print(f"REVIEW {spec[\'id\']} CUDA {epoch+1}/{spec[\'epochs\']} val_loss={val_loss:.5f} recall@.5={m[\'recall\']:.3f} {row[\'seconds\']:.1f}s",flush=True)\n    model.load_state_dict(best_weights);z=predictions(model,data[\'X_val\'],batch if image else 64,image);p=expit(z)\n    threshold=select_threshold(data[\'y_val\'],p);metrics=review_metrics(data[\'y_val\'],p,threshold)\n    np.savez_compressed(directory/\'validation_predictions.npz\',labels=data[\'y_val\'],probabilities=p,ids=data[\'ids_val\'])\n    result=dict(id=spec[\'id\'],kind=spec[\'kind\'],family=spec[\'family\'],route=spec[\'route\'],representation=contract[\'representation\'],\n        feature_shape=contract[\'shape\'],validation_ids_hash=contract[\'validation_ids_hash\'],train_ids_hash=contract[\'train_ids_hash\'],\n        seed=spec[\'seed\'],best_epoch=min(history,key=lambda h:h[\'val_loss\'])[\'epoch\'],epochs_completed=history[-1][\'epoch\'],\n        history=history,elapsed_new_training_seconds=time.monotonic()-started,\n        peak_gpu_allocated_bytes=torch.cuda.max_memory_allocated(),peak_gpu_reserved_bytes=torch.cuda.max_memory_reserved(),\n        gpu=torch.cuda.get_device_name(0),torch_version=torch.__version__,model_bytes=(directory/\'best.pt\').stat().st_size,\n        parameters=sum(p.numel() for p in model.parameters()),origin=spec.get(\'origin\'),continuation=origin_note,\n        test_set_evaluated=False,**metrics)\n    if not image:\n        save_keras_weights(model,directory/\'best_weights.npz\')\n        fixture=np.asarray(data[\'X_val\'][:4]);np.savez(directory/\'bridge_parity.npz\',features=fixture,logits=predictions(model,fixture,4,False))\n        settings=read_json(Path(spec[\'output\'])/\'settings.json\')\n        worker(settings,dict(action=\'bridge_restore\',id=spec[\'id\'],directory=str(directory/\'graph\'),\n                             weights=str(directory/\'best_weights.npz\'),model=str(directory/\'best.keras\')),framework=\'tf\')\n    save_json(directory/\'result.json\',result)\n    print(\'RESULT\',spec[\'id\'],json.dumps(metrics),flush=True)\n\n\ndef main():\n    """Accept only recorded JSON requests, keeping notebook kernels free of GPU allocations."""\n    parser=argparse.ArgumentParser(description=__doc__);parser.add_argument(\'--request\',type=Path,required=True)\n    args=parser.parse_args();train(read_json(args.request))\n\n\nif __name__==\'__main__\':main()\n', 'review_results.py': '"""Generate review tables, deployment measurements and five-slide evidence from saved runs."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom scripts.review_campaign import read_json,save_json,review_metrics,cohort_digest,worker\n\n\n\ndef recall_operating_point(labels,probabilities,target):\n    """Select the highest validation threshold meeting a declared RPW recall target."""\n    labels=np.asarray(labels);probabilities=np.asarray(probabilities)\n    if not 0<target<=1:raise ValueError(\'Recall target must be in (0,1]\')\n    positives=np.sort(probabilities[labels==1])[::-1]\n    if not len(positives):raise ValueError(\'RPW observations are required\')\n    required=int(np.ceil(target*len(positives)))\n    return review_metrics(labels,probabilities,float(positives[required-1]))\n\ndef append_codec(rows, output, name, model, train, val, labels, ids, threshold, model_bytes, representation):\n    """Measure real uint8 and PCA round trips with training-fitted profiles and frozen thresholds."""\n    from sklearn.decomposition import PCA\n    from scipy.special import expit\n    def evaluate(codec,recovered,size,profile):\n        """Evaluate only frozen validation observations after decoding actual integer codes."""\n        logits=np.concatenate([model(recovered[i:i+32,...,None],training=False).numpy().ravel() for i in range(0,len(recovered),32)])\n        p=expit(logits);metrics=review_metrics(labels,p,threshold)\n        identifier=name+\'_\'+codec\n        np.savez_compressed(output/\'plots\'/f\'{identifier}_predictions.npz\',labels=labels,probabilities=p,ids=ids)\n        np.savez_compressed(output/\'checkpoints\'/f\'{identifier}_codec.npz\',**profile)\n        row=dict(id=identifier,model=name,route=\'server\',representation=representation,codec=codec,\n                 model_bytes=model_bytes,feature_bytes_per_measurement=size,daily_report_bytes=None,\n                 scenario_measurements_per_day=12,scenario_feature_bytes_per_day=size*12,\n                 validation_ids_hash=cohort_digest(ids),board_verified=False,**metrics)\n        rows.append(row)\n    lo=train.min(axis=(0,1));hi=train.max(axis=(0,1));scale=np.maximum((hi-lo)/255,1e-8)\n    codes=np.clip(np.rint((val-lo)/scale),0,255).astype(\'uint8\')\n    recovered=np.frombuffer(codes.tobytes(),dtype=\'uint8\').reshape(val.shape).astype(\'float32\')*scale+lo\n    evaluate(\'uint8\',recovered,int(np.prod(val.shape[1:])),dict(lo=lo,scale=scale))\n    for size in [16,32]:\n        pca=PCA(n_components=size,svd_solver=\'full\').fit(train.reshape(len(train),-1))\n        projected=pca.transform(train.reshape(len(train),-1));low=projected.min(axis=0);high=projected.max(axis=0)\n        scale=np.maximum((high-low)/255,1e-8)\n        code=np.clip(np.rint((pca.transform(val.reshape(len(val),-1))-low)/scale),0,255).astype(\'uint8\')\n        decoded=np.frombuffer(code.tobytes(),dtype=\'uint8\').reshape(code.shape).astype(\'float32\')*scale+low\n        restored=pca.inverse_transform(decoded).reshape(val.shape).astype(\'float32\')\n        evaluate(f\'pca{size}\',restored,size,dict(mean=pca.mean_,components=pca.components_,lo=low,scale=scale))\n\n\ndef deployment(settings,results):\n    """Compare existing and newly selected local INT8/server models on identical evaluation IDs."""\n    import tensorflow as tf\n    import pandas as pd\n    from treevibes.export import tflite_predict\n    out=Path(settings[\'output\']);selection=read_json(out/\'audit/selection.json\');rows=[]\n    valid_hash=selection[\'cohort\'][\'val\'];cache=out/\'cache/mfcc13\';contract=read_json(cache/\'contract.json\')\n    train=((np.load(cache/\'X_train.npy\')-np.array(contract[\'mean\']))/np.array(contract[\'scale\'])).astype(\'float32\')\n    val=((np.load(cache/\'X_val.npy\')-np.array(contract[\'mean\']))/np.array(contract[\'scale\'])).astype(\'float32\')\n    labels=np.load(cache/\'y_val.npy\');ids=np.load(cache/\'ids_val.npy\')\n    previous=selection[\'server\'];source=Path(previous[\'run_dir\'])/\'best.keras\'\n    model=tf.keras.models.load_model(source,compile=False)\n    append_codec(rows,out,\'previous_\'+previous[\'id\'],model,train,val,labels,ids,previous[\'threshold\'],source.stat().st_size,\'mfcc13\')\n    tf.keras.backend.clear_session()\n    strongest=sorted([r for r in results if r[\'kind\'] in [\'feature_screen\',\'feature_full\'] and r[\'representation\']!=\'mfcc13\'],key=lambda r:(r[\'f1\'],r[\'recall\']),reverse=True)\n    if strongest:\n        r=strongest[0];cache=Path(r[\'cache\']);c=read_json(cache/\'contract.json\')\n        train=((np.load(cache/\'X_train.npy\')-np.array(c[\'mean\']))/np.array(c[\'scale\'])).astype(\'float32\')\n        val=((np.load(cache/\'X_val.npy\')-np.array(c[\'mean\']))/np.array(c[\'scale\'])).astype(\'float32\')\n        source=Path(r[\'directory\'])/\'best.keras\';model=tf.keras.models.load_model(source,compile=False)\n        append_codec(rows,out,r[\'id\'],model,train,val,labels,ids,r[\'threshold\'],source.stat().st_size,r[\'representation\'])\n        tf.keras.backend.clear_session()\n    # Existing INT8 exports are measured again from their actual bytes, including QAT.\n    candidates=[]\n    for meta in read_json(out/\'audit/original_exports.json\'):\n        if meta[\'method\']==\'float\':continue\n        parent=Path(meta[\'parent_run\']);cfg=read_json(parent/\'config.json\')\n        if cfg[\'target\']!=\'local\' or cfg[\'features\']!=contract[\'features\']:continue\n        location=Path(meta[\'directory\']);model_path=location/\'model.tflite\'\n        if not model_path.is_file() or hashlib.sha256(model_path.read_bytes()).hexdigest()!=meta[\'model_sha256\']:continue\n        raw=np.load(out/\'cache/mfcc13/X_val.npy\');input_c=read_json(parent/\'input_contract.json\')\n        from treevibes.train import normalized\n        p=tflite_predict(model_path,normalized(raw,input_c));m=review_metrics(labels,p,meta[\'threshold\'])\n        candidates.append((cfg[\'id\']+\'_\'+meta[\'method\'],model_path,m,p))\n    extended=out/\'checkpoints/extend_local\'\n    if (extended/\'result.json\').exists():\n        destination=extended/\'int8\'\n        if not (destination/\'export.json\').exists():\n            worker(settings,dict(action=\'int8\',id=\'extend_local_int8\',directory=str(destination),model=str(extended/\'best.keras\'),cache=str(out/\'cache/mfcc13\')),framework=\'tf\')\n        meta=read_json(destination/\'export.json\')\n        with np.load(destination/\'validation_predictions.npz\') as p:probs=p[\'probabilities\']\n        metrics={k:meta[k] for k in [\'accuracy\',\'precision\',\'recall\',\'f1\',\'pr_auc\',\'roc_auc\',\'confusion\',\'threshold\',\'count\',\'false_negatives\',\'false_positives\',\'fpr\']}\n        candidates.append((\'extend_local_int8\',destination/\'model.tflite\',metrics,probs))\n    for name,source,m,p in candidates:\n        np.savez_compressed(out/\'plots\'/f\'{name}_predictions.npz\',labels=labels,probabilities=p,ids=ids)\n        rows.append(dict(id=name,model=name,route=\'ESP32 candidate\',representation=\'mfcc13\',codec=\'local_int8\',\n                         model_bytes=source.stat().st_size,feature_bytes_per_measurement=0,daily_report_bytes=24,\n                         scenario_measurements_per_day=12,scenario_feature_bytes_per_day=24,\n                         validation_ids_hash=valid_hash,board_verified=False,**m))\n    for r in results:\n        if r[\'kind\']==\'compressed_server\':\n            import shutil\n            shutil.copy2(Path(r[\'directory\'])/\'validation_predictions.npz\',out/\'plots\'/f"{r[\'id\']}_predictions.npz")\n            rows.append(dict(id=r[\'id\'],model=r[\'id\'],route=\'server\',representation=r[\'representation\'],codec=\'pca16_direct\',\n                             model_bytes=r[\'model_bytes\'],feature_bytes_per_measurement=16,daily_report_bytes=None,\n                             scenario_measurements_per_day=12,scenario_feature_bytes_per_day=192,\n                             validation_ids_hash=valid_hash,board_verified=False,\n                             **{k:r[k] for k in [\'accuracy\',\'recall\',\'f1\',\'precision\',\'fpr\',\'false_negatives\',\'false_positives\',\'confusion\',\'threshold\',\'count\']}))\n    for r in results:\n        if r[\'kind\']==\'unrestricted\':\n            # The cached single-channel resized image is a measured float16 transport option.\n            # A smaller uint8 image is not assigned this score without a separate round-trip evaluation.\n            rows.append(dict(id=r[\'id\'],model=r[\'id\'],route=\'unrestricted server\',representation=r[\'representation\'],codec=\'float16_spectrogram\',\n                             model_bytes=r[\'model_bytes\'],feature_bytes_per_measurement=299*299*2,\n                             daily_report_bytes=None,scenario_measurements_per_day=12,scenario_feature_bytes_per_day=299*299*2*12,\n                             validation_ids_hash=valid_hash,board_verified=False,\n                             **{k:r[k] for k in [\'accuracy\',\'recall\',\'f1\',\'precision\',\'fpr\',\'false_negatives\',\'false_positives\',\'confusion\',\'threshold\',\'count\']}))\n    pd.DataFrame(rows).to_csv(out/\'deployment_comparison.csv\',index=False);save_json(out/\'deployment_comparison.json\',rows)\n    return rows\n\n\ndef render_plots(out, old, results,deploy):\n    """Write scientific charts and exact plotted data; retain actual healthy/RPW counts."""\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    plt.rcParams.update({\'font.size\':12,\'axes.spines.top\':False,\'axes.spines.right\':False,\'figure.dpi\':150})\n    plots=out/\'plots\'\n    examples=plots/\'same_recording_features.npz\'\n    if examples.exists():\n        with np.load(examples) as source:\n            fig,axes=plt.subplots(1,3,figsize=(11,2.6))\n            for ax,key,title in zip(axes,[\'mfcc\',\'mel\',\'pcen\'],[\'MFCC\',\'Log-Mel64\',\'PCEN64\']):\n                ax.imshow(source[key].T,aspect=\'auto\',origin=\'lower\',cmap=\'magma\');ax.set_title(title);ax.set_xlabel(\'Time frame\')\n            fig.tight_layout();fig.savefig(plots/\'feature_heatmaps.png\');plt.close(fig)\n    for route in [\'server\',\'local\']:\n        new=next((r for r in results if r[\'id\']==f\'extend_{route}\'),None)\n        if new:\n            previous=next(r for r in old if r[\'run_dir\']==new[\'origin\'])\n            history=read_json(out/\'audit/models\'/previous[\'id\']/\'training_result.json\').get(\'history\',[])\n            fig,ax=plt.subplots(figsize=(6.2,3.2))\n            for key,label,color in [(\'train_loss\',\'Train\',\'#64748b\'),(\'val_loss\',\'Validation\',\'#007f73\')]:\n                ax.plot([h[\'epoch\'] for h in new[\'history\']],[h[key] for h in new[\'history\']],label=label,color=color)\n            ax.axvline(max(h[\'epoch\'] for h in history),ls=\'--\',color=\'#d97706\',label=\'Previous stop\')\n            ax.set(xlabel=\'Epoch\',ylabel=\'Binary cross-entropy\',title=route.capitalize()+\' continuation\');ax.legend()\n            fig.tight_layout();fig.savefig(plots/f\'extension_{route}.png\');plt.close(fig)\n    local=max([r for r in deploy if r[\'route\']==\'ESP32 candidate\'],key=lambda r:(r[\'f1\'],r[\'recall\']))\n    server=max([r for r in deploy if r[\'route\']!=\'ESP32 candidate\'],key=lambda r:(r[\'f1\'],r[\'recall\']))\n    fig,axes=plt.subplots(1,2,figsize=(9,3.5))\n    for ax,r,title in zip(axes,[local,server],[\'Local INT8\',\'Best server\']):\n        counts=np.array(r[\'confusion\']);norm=counts/counts.sum(axis=1,keepdims=True)\n        ax.imshow(norm,vmin=0,vmax=1,cmap=\'Blues\')\n        for i in range(2):\n            for j in range(2):ax.text(j,i,f\'{counts[i,j]}\\n{norm[i,j]:.1%}\',ha=\'center\',va=\'center\',color=\'white\' if norm[i,j]>.6 else \'#102030\',fontsize=14)\n        ax.set(xticks=[0,1],yticks=[0,1],xticklabels=[\'healthy\',\'RPW\'],yticklabels=[\'healthy\',\'RPW\'],xlabel=\'Predicted\',ylabel=\'Actual\',title=title)\n    fig.tight_layout();fig.savefig(plots/\'finalist_confusion.png\');plt.close(fig)\n    from sklearn.metrics import precision_recall_curve\n    fig,ax=plt.subplots(figsize=(6,3.5))\n    for r in [local,server]:\n        path=plots/f"{r[\'id\']}_predictions.npz"\n        if not path.exists():path=Path(next(x for x in results if x[\'id\']==r[\'id\'])[\'directory\'])/\'validation_predictions.npz\'\n        with np.load(path) as p:precision,recall,thresholds=precision_recall_curve(p[\'labels\'],p[\'probabilities\'])\n        ax.plot(recall,precision,label=r[\'id\'])\n    ax.set(xlabel=\'RPW recall\',ylabel=\'RPW precision\',xlim=(0,1),ylim=(0,1));ax.legend(fontsize=8)\n    fig.tight_layout();fig.savefig(plots/\'finalist_precision_recall.png\');plt.close(fig)\n    return local,server\n\n\ndef summarize(settings,measure_deployment=True):\n    """Assemble tables and slide data exclusively from completed, cohort-matched evidence."""\n    import pandas as pd\n    out=Path(settings[\'output\']);selection=read_json(out/\'audit/selection.json\');old=read_json(out/\'audit/previous_results.json\')\n    results=[]\n    for path in (out/\'checkpoints\').glob(\'*/result.json\'):\n        r=read_json(path);spec=read_json(path.parent/\'config.json\')\n        if r[\'validation_ids_hash\']!=selection[\'cohort\'][\'val\']:raise ValueError(\'Finalist evaluation cohorts differ\')\n        r.update(directory=str(path.parent),cache=spec[\'cache\'],model_width=spec.get(\'model_config\',{}).get(\'width\'),split=\'validation\')\n        with np.load(path.parent/\'validation_predictions.npz\',allow_pickle=False) as pred:\n            expected_ids=np.load(out/\'cache/mfcc13/ids_val.npy\')\n            expected_labels=np.load(out/\'cache/mfcc13/y_val.npy\')\n            np.testing.assert_array_equal(pred[\'ids\'],expected_ids)\n            np.testing.assert_array_equal(pred[\'labels\'],expected_labels)\n            measured=review_metrics(pred[\'labels\'],pred[\'probabilities\'],r[\'threshold\'])\n            for key in [\'accuracy\',\'recall\',\'f1\',\'fpr\']:\n                if abs(measured[key]-r[key])>1e-10:raise ValueError(\'Recorded metric differs from saved predictions\')\n        results.append(r)\n    flat=[{k:v for k,v in r.items() if k not in [\'history\',\'continuation\',\'confusion\']} for r in results]\n    pd.DataFrame(flat).to_csv(out/\'runs.csv\',index=False)\n    features=[r for r in results if r[\'kind\'] in [\'feature_screen\',\'feature_full\']]\n    columns=[\'id\',\'family\',\'model_width\',\'kind\',\'representation\',\'feature_shape\',\'split\',\'seed\',\'accuracy\',\'recall\',\'f1\',\'fpr\',\'false_negatives\',\'false_positives\',\'best_epoch\',\'validation_ids_hash\']\n    pd.DataFrame(features).reindex(columns=columns).to_csv(out/\'feature_comparison.csv\',index=False)\n    deploy=deployment(settings,results) if measure_deployment else read_json(out/\'deployment_comparison.json\')\n    local,server=render_plots(out,old,results,deploy)\n    edge_high_recall=max([r for r in deploy if r[\'route\']==\'ESP32 candidate\'],key=lambda r:(r[\'recall\'],r[\'f1\'],-r[\'model_bytes\']))\n    operating=[]\n    for finalist in [local,server]:\n        prediction=out/\'plots\'/f"{finalist[\'id\']}_predictions.npz"\n        if not prediction.exists():prediction=Path(next(r for r in results if r[\'id\']==finalist[\'id\'])[\'directory\'])/\'validation_predictions.npz\'\n        with np.load(prediction) as p:\n            for target in [.95,.98,1.]:\n                operating.append(dict(id=finalist[\'id\'],target_recall=target,**recall_operating_point(p[\'labels\'],p[\'probabilities\'],target)))\n    pd.DataFrame(operating).to_csv(out/\'operating_points.csv\',index=False)\n    extensions=[]\n    for route in [\'server\',\'local\']:\n        new=next((r for r in results if r[\'id\']==f\'extend_{route}\'),None)\n        if new:extensions.append(dict(route=route,old=selection[route],new=new))\n    screen=[r for r in features if r[\'kind\']==\'feature_screen\'];full=[r for r in features if r[\'kind\']==\'feature_full\']\n    feature_best=max(features,key=lambda r:(r[\'f1\'],r[\'recall\'])) if features else None\n    unrestricted=next((r for r in results if r[\'kind\']==\'unrestricted\'),None)\n    compressed=max([r for r in deploy if r[\'codec\'].startswith(\'pca16\')],key=lambda r:(r[\'f1\'],r[\'recall\']))\n    full_mfcc=next(r for r in deploy if r[\'representation\']==\'mfcc13\' and r[\'codec\']==\'uint8\')\n    delta=max((e[\'new\'][\'f1\']-e[\'old\'][\'f1\'] for e in extensions),default=0)\n    lines=[\'# Final model decisions\',\'\',f"Original campaign: {len(old)} canonical recipes; {len(results)} completed review runs.",\n        \'All scores below are validation results on the same 361 accepted recordings (109 RPW, 252 healthy).\',\n        \'The final test set was not evaluated. Thresholds selected on validation; file/duplicate grouping, not tree/session isolation.\',\'\',\n        f"- Extra training: maximum F1 change {delta*100:+.2f} percentage points; select the minimum-validation-loss checkpoint.",\n        f"- Best balanced edge candidate: {local[\'id\']}, {local[\'model_bytes\']} bytes, recall {local[\'recall\']:.2%}, F1 {local[\'f1\']:.2%}.",\n        f"- Best measured deployment-server route: {server[\'id\']}, recall {server[\'recall\']:.2%}, F1 {server[\'f1\']:.2%}.",\n        f"- Compact server payload: {compressed[\'id\']}, 16 bytes/measurement, recall {compressed[\'recall\']:.2%}, F1 {compressed[\'f1\']:.2%}.",\n        \'- Every node sends at least one daily report. Local 24-byte summary is a proposed prediction/health report; remote features retain each reading.\',\n        \'- ESP32 end-to-end RAM, latency, firmware size and power remain unmeasured. Radio framing, security and retries are extra.\',\n        \'- Image-based unrestricted inference is an accuracy experiment; its large input is not a LoRa deployment recommendation.\',\'\',\n        \'## Uncompleted/omitted menu items\',\'\',\n        \'- Separate lower-initial-learning-rate ablation, raw-waveform CNN, extra seeds, cross-feature knowledge distillation and acoustic-noise tests were not automatically launched; one volume perturbation was completed separately.\',\n        \'- Previous training histories contain losses; newly recorded epochs also contain recall/F1 at threshold 0.5. Missing old epoch metrics are not invented.\']\n    if feature_best:lines.insert(8,f"- Best feature trial by F1: {feature_best[\'id\']}, F1 {feature_best[\'f1\']:.2%}, recall {feature_best[\'recall\']:.2%}.")\n    if unrestricted:lines.insert(9,f"- Unrestricted InceptionResNet-V2: accuracy {unrestricted[\'accuracy\']:.2%}, recall {unrestricted[\'recall\']:.2%}, F1 {unrestricted[\'f1\']:.2%}.")\n    (out/\'final_conclusions.md\').write_text(\'\\n\'.join(lines)+\'\\n\',encoding=\'utf-8\')\n    previous_accuracy=max([r for r in old if r[\'status\']==\'completed\' and r[\'same_evaluation_cohort\']],key=lambda r:(r[\'accuracy\'],r[\'recall\']))\n    feature_accuracy=max(features,key=lambda r:(r[\'accuracy\'],r[\'recall\'])) if features else None\n    ceiling=max([previous_accuracy]+results,key=lambda r:(r[\'accuracy\'],r[\'recall\']))\n    lines.insert(7,f"- Highest measured accuracy on this cohort: {ceiling[\'id\']}, {ceiling[\'accuracy\']:.2%}, RPW recall {ceiling[\'recall\']:.2%}.")\n    (out/\'final_conclusions.md\').write_text(\'\\n\'.join(lines)+\'\\n\',encoding=\'utf-8\')\n    volume=read_json(out/\'volume_check.json\') if (out/\'volume_check.json\').exists() else None\n    if volume:\n        for row in volume[\'results\']:\n            if row[\'perturbation\']==\'audio_gain_0_5\':\n                lines.insert(8,f"- Half-volume waveform check, fixed thresholds: {row[\'model\']} produced {row[\'false_positives\']} healthy false alarms ({row[\'fpr\']:.2%}); RPW recall {row[\'recall\']:.2%}.")\n        lines.insert(8,\'- Gain sensitivity needs mitigation and re-evaluation before field deployment; the volume check does not establish field performance.\')\n        (out/\'final_conclusions.md\').write_text(\'\\n\'.join(lines)+\'\\n\',encoding=\'utf-8\')\n    lines.insert(12,f"- High-recall edge alternative: {edge_high_recall[\'id\']}, {edge_high_recall[\'model_bytes\']} bytes, RPW recall {edge_high_recall[\'recall\']:.2%}, F1 {edge_high_recall[\'f1\']:.2%}, {edge_high_recall[\'false_positives\']} healthy false alarms.")\n    (out/\'final_conclusions.md\').write_text(\'\\n\'.join(lines)+\'\\n\',encoding=\'utf-8\')\n    payload=dict(edge_high_recall=edge_high_recall,volume_check=volume,evaluation=\'validation\',count=selection[\'validation_count\'],rpw=109,healthy=252,cohort=selection[\'cohort\'][\'val\'],\n                 extensions=extensions,features=screen,feature_full=full,feature_best=feature_best,\n                 previous_server=selection[\'server\'],previous_accuracy=previous_accuracy,feature_accuracy=feature_accuracy,accuracy_ceiling=ceiling,unrestricted=unrestricted,local=local,server=server,\n                 operating_points=operating,deployment=[local,full_mfcc,compressed]+([next(r for r in deploy if r[\'route\']==\'unrestricted server\')] if unrestricted else []))\n    save_json(out/\'slide_data.json\',payload)\n    save_json(out/\'report_status.json\',dict(completed_review_runs=len(results),test_set_evaluated=False,slide_data_ready=bool(extensions and screen and unrestricted)))\n    print(\'REPORT READY\',out,flush=True)\n    return payload\n\n\ndef main():\n    """Regenerate recorded results without training or changing checkpoints."""\n    parser=argparse.ArgumentParser(description=__doc__);parser.add_argument(\'--settings\',type=Path,required=True)\n    parser.add_argument(\'--reuse-deployment\',action=\'store_true\');args=parser.parse_args()\n    summarize(read_json(args.settings),not args.reuse_deployment)\n\n\nif __name__==\'__main__\':main()\n', 'review_compressed.py': '"""One direct classifier on actual 16-byte PCA codes, rather than reconstructing MFCC first."""\nimport argparse\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom scripts.review_campaign import read_json,save_json,cohort_digest,review_metrics\n\n\ndef run(settings):\n    """Fit a fixed histogram booster on train-only PCA16 uint8 codes and measure held-out validation."""\n    import joblib\n    import sklearn\n    from sklearn.ensemble import HistGradientBoostingClassifier\n    from treevibes.evaluate import select_threshold\n    out=Path(settings[\'output\']);directory=out/\'checkpoints/server_pca16_histogram\';directory.mkdir(exist_ok=True)\n    if (directory/\'result.json\').exists():return\n    selection=read_json(out/\'audit/selection.json\');cache=out/\'cache/mfcc13\';contract=read_json(cache/\'contract.json\')\n    profile=out/\'checkpoints\'/(\'previous_\'+selection[\'server\'][\'id\']+\'_pca16_codec.npz\')\n    with np.load(profile) as p:parameters={k:p[k] for k in p.files}\n    codes={};ids={};labels={}\n    for part in [\'train\',\'val\']:\n        raw=np.load(cache/f\'X_{part}.npy\');ids[part]=np.load(cache/f\'ids_{part}.npy\');labels[part]=np.load(cache/f\'y_{part}.npy\')\n        x=((raw-np.array(contract[\'mean\']))/np.array(contract[\'scale\'])).astype(\'float32\').reshape(len(raw),-1)\n        projected=(x-parameters[\'mean\'])@parameters[\'components\'].T\n        q=np.clip(np.rint((projected-parameters[\'lo\'])/parameters[\'scale\']),0,255).astype(\'uint8\')\n        codes[part]=np.stack([np.frombuffer(row.tobytes(),dtype=\'uint8\') for row in q])\n        assert codes[part].shape==(len(raw),16)\n    config=dict(id=\'server_pca16_histogram\',kind=\'compressed_server\',family=\'histogram\',seed=42,\n                cache=str(cache),codec=str(profile),feature_bytes=16,max_iter=200,max_leaf_nodes=31,\n                learning_rate=.1,l2_regularization=0.,early_stopping=True,internal_validation_fraction=.1,\n                sklearn_version=sklearn.__version__)\n    save_json(directory/\'config.json\',config);started=time.monotonic()\n    model=HistGradientBoostingClassifier(max_iter=200,max_leaf_nodes=31,learning_rate=.1,random_state=42,\n        early_stopping=True,validation_fraction=.1,n_iter_no_change=15)\n    model.fit(codes[\'train\'],labels[\'train\']);p=model.predict_proba(codes[\'val\'])[:,1]\n    threshold=select_threshold(labels[\'val\'],p);metrics=review_metrics(labels[\'val\'],p,threshold)\n    joblib.dump(model,directory/\'best.joblib\')\n    np.testing.assert_allclose(joblib.load(directory/\'best.joblib\').predict_proba(codes[\'val\'])[:,1],p,rtol=0,atol=0)\n    np.savez_compressed(directory/\'codec.npz\',**parameters)\n    np.savez_compressed(directory/\'validation_predictions.npz\',labels=labels[\'val\'],probabilities=p,ids=ids[\'val\'])\n    history=[dict(iteration=i,train_loss=float(-a),internal_train_holdout_loss=float(-b))\n             for i,(a,b) in enumerate(zip(model.train_score_,model.validation_score_))]\n    save_json(directory/\'history.json\',history)\n    result=dict(id=config[\'id\'],kind=config[\'kind\'],family=config[\'family\'],route=\'server\',representation=\'pca16_uint8\',\n                feature_shape=[16],validation_ids_hash=cohort_digest(ids[\'val\']),train_ids_hash=cohort_digest(ids[\'train\']),\n                seed=42,best_epoch=None,epochs_completed=model.n_iter_,history=history,\n                elapsed_new_training_seconds=time.monotonic()-started,model_bytes=(directory/\'best.joblib\').stat().st_size,\n                parameters=None,codec=str(directory/\'codec.npz\'),test_set_evaluated=False,**metrics)\n    save_json(directory/\'result.json\',result);print(\'DIRECT PCA16 SERVER\',metrics,flush=True)\n\n\ndef main():\n    """Run this small CPU experiment separately from the active GPU controller."""\n    parser=argparse.ArgumentParser(description=__doc__);parser.add_argument(\'--settings\',type=Path,required=True)\n    args=parser.parse_args();run(read_json(args.settings))\n\n\nif __name__==\'__main__\':main()\n', 'review_robustness.py': '"""One bounded audio-volume check on fixed validation finalists; no training or threshold tuning."""\nimport argparse\nfrom pathlib import Path\nimport numpy as np\nfrom scripts.review_campaign import read_json,save_json,resolve_audio,review_metrics\n\n\ndef mfcc_with_gain(path,gain,features):\n    """Re-extract the baseline MFCC after gain, retaining real log floors and dynamic-range clipping."""\n    import librosa\n    from scipy.signal import butter,filtfilt\n    c=features;signal,_=librosa.load(path,sr=c[\'sample_rate\'],mono=True)\n    length=int(c[\'seconds\']*c[\'sample_rate\']);signal=np.pad(signal[:length],(0,max(0,length-len(signal))))\n    b,a=butter(c[\'filter_order\'],[c[\'low_hz\'],c[\'high_hz\']],btype=\'bandpass\',fs=c[\'sample_rate\'])\n    filtered=filtfilt(b,a,signal)*gain;fft=length//c[\'frames\']\n    mel=librosa.feature.melspectrogram(y=filtered,sr=c[\'sample_rate\'],n_fft=fft,hop_length=fft,\n        win_length=fft,window=c[\'window\'],center=False,power=c[\'power\'],n_mels=c[\'n_mels\'],\n        fmin=c[\'fmin\'],fmax=c[\'fmax\'],htk=False,norm=c[\'mel_norm\'])\n    log=librosa.power_to_db(mel,ref=1.,amin=c[\'log_amin\'],top_db=c[\'log_top_db\'])\n    return librosa.feature.mfcc(S=log,n_mfcc=c[\'coefficients\'],dct_type=c[\'dct_type\'],norm=c[\'dct_norm\'],lifter=0).T.astype(\'float32\')\n\n\ndef run(settings):\n    """Compare half-volume with unmodified audio on the existing accepted validation cohort."""\n    import tensorflow as tf\n    import pandas as pd\n    from scipy.special import expit\n    from treevibes.export import tflite_predict\n    from treevibes.train import normalized\n    out=Path(settings[\'output\']);selection=read_json(out/\'audit/selection.json\')\n    c=read_json(out/\'cache/mfcc13/contract.json\');root=Path(read_json(out/\'audit/audio_source.json\')[\'root\'])\n    ids=np.load(out/\'cache/mfcc13/ids_val.npy\');labels=np.load(out/\'cache/mfcc13/y_val.npy\')\n    original=np.load(out/\'cache/mfcc13/X_val.npy\')\n    # Verify the independent extraction path before assigning any robustness result.\n    for i in range(3):np.testing.assert_allclose(mfcc_with_gain(resolve_audio(root,str(ids[i])),1.,c[\'features\']),original[i],rtol=1e-5,atol=1e-4)\n    quieter=np.stack([mfcc_with_gain(resolve_audio(root,str(identifier)),.5,c[\'features\']) for identifier in ids])\n    previous=selection[\'server\'];parent=Path(previous[\'run_dir\']);contract=read_json(parent/\'input_contract.json\')\n    model=tf.keras.models.load_model(parent/\'best.keras\',compile=False)\n    exports=read_json(out/\'audit/original_exports.json\')\n    local=max([r for r in exports if r[\'method\']==\'ptq\' and \'tiny_temporal_w12\' in r[\'parent_run\']],key=lambda r:r[\'validation\'][\'f1\'])\n    local_parent=Path(local[\'parent_run\']);local_contract=read_json(local_parent/\'input_contract.json\')\n    rows=[]\n    for name,raw in [(\'original_gain_1\',original),(\'audio_gain_0_5\',quieter)]:\n        x=normalized(raw,contract)\n        probabilities=expit(np.concatenate([model(x[i:i+64,...,None],training=False).numpy().ravel() for i in range(0,len(x),64)]))\n        rows.append(dict(model=previous[\'id\'],perturbation=name,**review_metrics(labels,probabilities,previous[\'threshold\'])))\n        local_p=tflite_predict(Path(local[\'directory\'])/\'model.tflite\',normalized(raw,local_contract))\n        rows.append(dict(model=\'tiny_temporal_w12_ptq\',perturbation=name,**review_metrics(labels,local_p,local[\'threshold\'])))\n    pd.DataFrame(rows).to_csv(out/\'volume_check.csv\',index=False)\n    save_json(out/\'volume_check.json\',dict(results=rows,thresholds_retuned=False,gating_reapplied=False,\n        validation_ids_hash=selection[\'cohort\'][\'val\'],test_set_evaluated=False,\n        scope=\'one synthetic waveform gain perturbation; not a microphone/noise/field validation\'))\n    print(\'VOLUME CHECK\',[(r[\'model\'],r[\'perturbation\'],r[\'recall\'],r[\'f1\']) for r in rows],flush=True)\n\n\ndef main():\n    """Run one optional CPU diagnostic without consuming GPU training memory."""\n    parser=argparse.ArgumentParser(description=__doc__);parser.add_argument(\'--settings\',type=Path,required=True)\n    args=parser.parse_args();run(read_json(args.settings))\n\n\nif __name__==\'__main__\':main()\n'}
if not ACTIVE:
    (OUTPUT / "settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
    package = OUTPUT / "tools/scripts"
    (package / "__init__.py").write_text("from pkgutil import extend_path\n__path__=extend_path(__path__,__name__)\n")
    for name, source in sources.items():
        (package / name).write_text(source, encoding="utf-8")
ENV = os.environ.copy()
ENV.update(PYTHONPATH=os.pathsep.join([str(OUTPUT/"tools"), str(ROOT), str(OUTPUT/"dependencies")]),
    PYTHONUTF8="1", PYTHONIOENCODING="utf-8", MPLBACKEND="Agg", TF_USE_LEGACY_KERAS="1",
    TF_NUM_INTEROP_THREADS="1", TF_NUM_INTRAOP_THREADS="1", HF_HOME=str(OUTPUT/"pretrained_cache"))
sys.path.insert(0, str(ROOT))
from scripts.windows_support import run_logged
if not ACTIVE:
    extras = OUTPUT / "dependencies"
    probe = subprocess.run([str(GPU_PYTHON), "-c", "import torch,torchvision,timm,huggingface_hub,safetensors; assert torch.cuda.is_available(); print(torch.__version__, torchvision.__version__, timm.__version__)"], env=ENV, capture_output=True, text=True)
    if probe.returncode:
        run_logged([str(GPU_PYTHON), "-m", "pip", "install", "--target", str(extras), "--no-deps",
            "torchvision==0.22.1", "--index-url", "https://download.pytorch.org/whl/cu128"], OUTPUT/"logs/torchvision_install.log", stream=True)
        run_logged([str(GPU_PYTHON), "-m", "pip", "install", "--target", str(extras), "--no-deps",
            "timm==1.0.15", "huggingface-hub==0.30.2", "safetensors==0.5.3"], OUTPUT/"logs/review_dependencies.log", stream=True)
print("Review tools ready; original project files unchanged.")


## 3. Start or attach

Two optimizer continuations → core feature screens → unrestricted InceptionResNet-V2 → remaining screens → two larger feature trials.

In [ ]:
if ACTIVE:
    print("Attached to the running review. No duplicate worker was started.")
elif progress.get("state") in {"completed", "completed_with_failures"}:
    print("Review training already finished; continue to the report cell.")
else:
    log = (OUTPUT / "campaign.log").open("a", encoding="utf-8")
    process = subprocess.Popen([str(TF_PYTHON), str(OUTPUT/"tools/scripts/review_campaign.py"),
        "--settings", str(OUTPUT/"settings.json")], cwd=ROOT, env=ENV, stdout=log, stderr=subprocess.STDOUT)
    (OUTPUT/"controller.json").write_text(json.dumps(dict(pid=process.pid)))
    log.close()
    print("Started review controller PID", process.pid)
print("Rerun the next cell to view progress. Training survives closing this notebook.")


## 4. Monitor durable checkpoints

In [ ]:
progress = json.loads(progress_path.read_text()) if progress_path.exists() else {}
print(progress)
log = OUTPUT / "campaign.log"
if log.exists(): print(log.read_text(encoding="utf-8", errors="replace")[-5000:])
print("Saved results:", len(list((OUTPUT/"checkpoints").glob("*/result.json"))))


## 5. Measure deployment routes and prepare slide evidence

Run after training finishes. Reports preserve exact RPW misses and healthy false alarms. A 24-byte local daily report is proposed; server features are required for every measurement. ESP32 board memory/latency remain unmeasured.

In [ ]:
progress = json.loads(progress_path.read_text()) if progress_path.exists() else {}
if progress.get("state") not in {"completed", "completed_with_failures"}:
    print("Training is still running. Rerun this cell after completion.")
else:
    if not list((OUTPUT/"checkpoints").glob("previous_*_pca16_codec.npz")):
        run_logged([str(TF_PYTHON), str(OUTPUT/"tools/scripts/review_results.py"),
            "--settings", str(OUTPUT/"settings.json")], OUTPUT/"logs/initial_report.log", env=ENV, cwd=ROOT)
    for tool, receipt in [("review_compressed.py", "checkpoints/server_pca16_histogram/result.json"),
                          ("review_robustness.py", "volume_check.json")]:
        if not (OUTPUT/receipt).exists():
            run_logged([str(TF_PYTHON), str(OUTPUT/"tools/scripts"/tool), "--settings", str(OUTPUT/"settings.json")],
                OUTPUT/"logs"/(tool+".log"), env=ENV, cwd=ROOT)
    run_logged([str(TF_PYTHON), str(OUTPUT/"tools/scripts/review_results.py"),
        "--settings", str(OUTPUT/"settings.json")], OUTPUT/"logs/report.log", env=ENV, cwd=ROOT)
    from IPython.display import Markdown, display, Image
    display(Markdown((OUTPUT/"final_conclusions.md").read_text(encoding="utf-8")))
    display(Image(filename=str(OUTPUT/"plots/finalist_confusion.png")))
    print("Tables, source predictions, slide_data.json and plots:", OUTPUT)
    print("Editable PowerPoint/PDF are exported from these measured outputs by the accompanying deck builder.")
